## 五、高级特性（⭐ 全路线最重要的模块之一）

| 类别 | 知识点 |
|------|--------|
| 迭代器 | 可迭代对象 vs 迭代器、`__iter__` / `__next__`、iter() / next() |
| 生成器 | yield、生成器表达式、yield from、惰性求值 |
| 装饰器 | 闭包回顾、`@` 语法、带参数的装饰器、`functools.wraps` |
| 上下文管理器 | with 语句、`__enter__` / `__exit__`、contextlib |
| 函数式工具 | functools.partial、lru_cache |

⭐ **学习重点**
- **生成器 = Streaming 的底层**：LLM 逐 token 输出（`stream()` / `astream()`）就是生成器在驱动，必须亲手写几个 yield 例子。
- **装饰器**：`@tool`、`@retry`、`@cache` 等框架语法全是装饰器，要能自己写一个带参数的装饰器。
- **上下文管理器**：管理模型客户端、数据库连接、文件句柄的标准姿势。
- **迭代器协议**：看懂"为什么 for 能遍历一切"。

⚠️ 常见坑：生成器只能遍历一次、装饰器忘加 `functools.wraps` 丢失函数名。

### 1. 迭代器（Iterator）
- 可迭代对象 Iterable：可以被 for 循环遍历的对象
- 迭代器 Iterator：可以被 next() 函数调用并不断返回下一个值的对象

#### 1.1 迭代器
- 迭代器是有状态的，可以记住遍历的位置，通常只能遍历一次
- 每次调用 next() 函数返回迭代器的下一个值，直到没有更多值时抛出 StopIteration 异常

In [14]:
# list 是可迭代对象，iter(list)返回迭代器
lst = iter([1, 2, 3])
print(next(lst))
print(next(lst))
print(next(lst))

1
2
3


In [3]:
print(next(lst))

StopIteration: 

Iterator 和 list 的区别
- list： 数据已经全部存在内存中，可以重复遍历
- Iterator：按需取值，通常只能遍历一次

#### 1.2 for循环本质
- iter(obj) 转迭代器 -> 不断 next() -> StopIteration停止

In [5]:
# for循环本质：iter(obj) 转迭代器 -> 不断 next() -> StopIteration停止
lst = iter([1, 2, 3])
while True:
    try:
        x = next(lst)
        print(x)
    except StopIteration:
        break

1
2
3


#### 1.3 自定义迭代器

迭代器协议 Iterator Protocol：`__iter__` / `__next__`
   - `__iter__` 返回迭代器对象
   - `__next__` 返回下一个值，没有更多值时抛出 StopIteration 异常

In [15]:
class Counter:
    def __init__(self, n):
        self.n = n
        self.current = 0

    def __iter__(self): # 返回自己
        return self

    def __next__(self): # 返回下一个值
        if self.current >= self.n:
            raise StopIteration
        value = self.current
        self.current += 1
        return value

In [11]:
c = Counter(3)
print(next(c))
print(next(c))
print(next(c))

0
1
2


In [10]:
for x in Counter(3):
    print(x)

0
1
2


判断一个对象是不是Iterator

In [13]:
from collections.abc import Iterable, Iterator
lst = [1, 2, 3]
it = iter(lst)
print(isinstance(lst, Iterable))
print(isinstance(lst, Iterator))
print(isinstance(it, Iterable))
print(isinstance(it, Iterator))

True
False
True
True


### 2. 生成器（Generator）

#### 2.1 两种创建方式
- 生成器函数
- 生成器推导式

1. 生成器函数

In [18]:
def generator_function():
    yield 1
    yield 2
    yield 3

g = generator_function()

print(next(g))
print(next(g))
print(next(g))

1
2
3


2. 生成器推导式

In [21]:
g = (i*i for i in range(3))
print(next(g))
print(next(g))
print(next(g))



0
1
4
